# Data Cleaning & Labeling
In this notebook, I'll clean and label the dataset downloaded from ACLED's "Data Export Tool", producing a tailored dataset to better serve my research topic.

## 1. Remove Useless Data
For analytical convenience, I remove Hong Kong & Macao data in the original dataset. Also, I deleted some columns that are useless for my topic.

### 1.1 Remove Hong Kong & Macao Data

In [42]:
# Clean Hong Kong and Macao data
# Filter out all rows where admin1 is Hong Kong or Macao/Macau

import csv
from pathlib import Path

src = Path("../Data/ACLED_China_riot&protest_raw.csv")
dst = Path("../Data/cleaned_data.csv")

with src.open(newline="", encoding="utf-8-sig") as fin:
    reader = csv.DictReader(fin)
    rows = list(reader)
    fieldnames = reader.fieldnames

blocked_admin1 = {"hong kong", "macao", "macau"}
removed_admin1_values = sorted({
    (r.get("admin1") or "").strip()
    for r in rows
    if (r.get("admin1") or "").strip().lower() in blocked_admin1
})
kept = [
    r for r in rows
    if (r.get("admin1") or "").strip().lower() not in blocked_admin1
]

with dst.open("w", newline="", encoding="utf-8") as fout:
    writer = csv.DictWriter(fout, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(kept)

print(f"Total rows (excluding header): {len(rows)}")
print(f"Admin1 values removed: {removed_admin1_values}")
print(f"Rows removed: {len(rows)-len(kept)}")
print(f"Rows kept: {len(kept)}")
print(f"Output file: {dst}")

# Validation check
# The following confirms no rows still mention Hong Kong or Macao/Macau in location hierarchy fields.


keywords = ["hong kong", "macao", "macau"]
p = Path("../Data/cleaned_data.csv")
with p.open(newline="", encoding="utf-8") as f:
    r = csv.DictReader(f)
    bad = 0
    for row in r:
        txt = " | ".join([
            row.get("admin1", ""),
            row.get("admin2", ""),
            row.get("admin3", ""),
            row.get("location", "")
        ]).lower()
        if any(k in txt for k in keywords):
            bad += 1

print("Rows still mentioning Hong Kong/Macao in admin/location fields:", bad)

Total rows (excluding header): 12437
Admin1 values removed: ['Hong Kong', 'Macao']
Rows removed: 3500
Rows kept: 8937
Output file: ../Data/cleaned_data.csv
Rows still mentioning Hong Kong/Macao in admin/location fields: 0


### 1.2 Remove Region & Country Name

In [43]:
import csv
from pathlib import Path

file_path = Path("../Data/cleaned_data.csv")
temp_path = file_path.with_suffix(".tmp.csv")

with file_path.open(newline="", encoding="utf-8") as fin:
    reader = csv.DictReader(fin)
    rows = list(reader)
    original_fields = reader.fieldnames or []

drop_fields = {"iso", "region", "country"}
existing_lower = {f.lower() for f in original_fields}
kept_fields = [f for f in original_fields if f.lower() not in drop_fields]
removed_fields = [f for f in original_fields if f.lower() in drop_fields]
already_absent = sorted([c for c in drop_fields if c not in existing_lower])

if not original_fields:
    raise ValueError("cleaned_data.csv has no header. Please rerun Cell 4 first to regenerate it.")
if not kept_fields:
    raise ValueError("No columns would remain after dropping fields. Check the source file.")

# Write to a temp file first so a runtime error cannot partially overwrite the dataset.
with temp_path.open("w", newline="", encoding="utf-8") as fout:
    writer = csv.DictWriter(fout, fieldnames=kept_fields, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rows)

temp_path.replace(file_path)

print(f"Original columns: {len(original_fields)}")
if removed_fields:
    print(f"Columns removed this run: {removed_fields}")
else:
    print("No matching columns were removed in this run (they were already absent).")

print(f"Target columns already absent before write: {already_absent}")
print(f"Remaining columns: {len(kept_fields)}")
print(f"Rows preserved: {len(rows)}")
print(f"Updated file: {file_path}")

Original columns: 31
Columns removed this run: ['iso', 'region', 'country']
Target columns already absent before write: []
Remaining columns: 28
Rows preserved: 8937
Updated file: ../Data/cleaned_data.csv


## 2. Recode Crowd Size Data
In the original dataset, protest crowd size in the 'Tags" column are coded in an untidy way, with inconsistent labeling for the same value. For example, for crowd size between 101 and 1000, there's both 'crowd size=101-1000' and 'crowd size=between 101 and 1000'. Moreover, there's overly vague statement such as 'crowd size=a large number'. To facilitate a rigor analysis, these data must be recoded.


In [44]:
# Summary of crowd size labels in the original dataset
import csv
import re
from collections import Counter
from pathlib import Path

file_path = Path("../Data/cleaned_data.csv")

with file_path.open(newline="", encoding="utf-8") as fin:
    reader = csv.DictReader(fin)
    rows = list(reader)

field_map = {f.lower(): f for f in (reader.fieldnames or [])}
tags_col = field_map.get("tags")
if not tags_col:
    raise ValueError("No 'Tags' column found in cleaned_data.csv")

crowd_pattern = re.compile(r"crowd\s*size\s*=\s*([^;|,]+)", re.IGNORECASE)

raw_label_counter = Counter()
for row in rows:
    tag_text = (row.get(tags_col) or "").strip()
    if not tag_text:
        continue
    matches = crowd_pattern.findall(tag_text)
    for m in matches:
        raw_label = " ".join(m.split()).lower()
        raw_label_counter[raw_label] += 1


def normalize_crowd_label(label: str) -> str:
    text = label.lower().strip()

    if re.search(r"\b(1-10|between\s*1\s*and\s*10|up\s*to\s*10)\b", text):
        return "1-10"
    if re.search(r"\b(11-50|between\s*11\s*and\s*50)\b", text):
        return "11-50"
    if re.search(r"\b(51-100|between\s*51\s*and\s*100)\b", text):
        return "51-100"
    if re.search(r"\b(101-1000|between\s*101\s*and\s*1000)\b", text):
        return "101-1000"
    if re.search(r"\b(1000\+|more\s*than\s*1000|over\s*1000)\b", text):
        return "1000+"

    if re.search(r"\b(large\s*number|hundreds|thousands|many|massive|big)\b", text):
        return "vague_large"
    if re.search(r"\b(small|few|dozens)\b", text):
        return "vague_small"

    return "other_unclear"

normalized_counter = Counter()
for label, cnt in raw_label_counter.items():
    normalized_counter[normalize_crowd_label(label)] += cnt

rows_with_tags = sum(1 for r in rows if (r.get(tags_col) or "").strip())
rows_with_crowd_size = sum(raw_label_counter.values())

print("=== Crowd Size Summary (from Tags) ===")
print(f"Total rows: {len(rows)}")
print(f"Rows with non-empty Tags: {rows_with_tags}")
print(f"Rows/mentions with crowd size labels: {rows_with_crowd_size}")
print(f"Unique raw crowd size labels: {len(raw_label_counter)}")

print("\nTop raw crowd size labels:")
for label, cnt in raw_label_counter.most_common(20):
    print(f"- {label}: {cnt}")

print("\nNormalized crowd size groups:")
for label, cnt in normalized_counter.most_common():
    print(f"- {label}: {cnt}")

=== Crowd Size Summary (from Tags) ===
Total rows: 8937
Rows with non-empty Tags: 8937
Rows/mentions with crowd size labels: 8937
Unique raw crowd size labels: 478

Top raw crowd size labels:
- no report: 3856
- as many as 100: 1496
- up to 100: 734
- over 100: 186
- more than 100: 151
- dozens: 109
- hundreds: 100
- less than 100: 87
- 101-1000: 83
- unreported: 60
- around 100: 60
- a few hundred: 57
- 1-100: 52
- more than 20: 43
- between 1 to 100: 41
- a group: 40
- thousands: 38
- at least 3: 36
- more than 200: 34
- more than 10: 31

Normalized crowd size groups:
- other_unclear: 6820
- vague_large: 1757
- vague_small: 215
- 101-1000: 119
- 1000+: 23
- 1-10: 3


In [45]:
# Recode all crowd-size labels and export to CSV
import csv
import re
from collections import defaultdict
from pathlib import Path

if "raw_label_counter" not in globals():
    raise RuntimeError("Please run Cell 8 first so raw_label_counter is available.")


def canonicalize_label(label: str) -> str:
    t = label.lower().strip()
    t = t.replace("–", "-").replace("—", "-")
    t = re.sub(r"\s+", " ", t)

    # Common semantic synonyms
    if t in {
        "no report",
        "unreported",
        "not reported",
        "none reported",
        "unknown",
        "no reports",
        "no reported",
        "-no report",
        "unstated",
    }:
        return "no report"

    # Normalize common wording while preserving meaning
    t = re.sub(r"\bbetween\b", "", t)
    t = re.sub(r"\bfrom\b", "", t)
    t = re.sub(r"\babout\b|\baround\b|\bapproximately\b|\bapprox\.?\b|\balmost\b|\bnearly\b", "~", t)
    t = re.sub(r"\bclose\s*-?\s*", "~", t)
    t = re.sub(r"\bat least\b", ">=", t)
    t = re.sub(r"\bmore than\b|\bover\b|\bor more\b", ">", t)
    t = re.sub(r"\bless than\b", "<", t)
    t = re.sub(r"\bup to\b", "<=", t)
    t = re.sub(r"\bas many as\b", "<=", t)
    t = re.sub(r"\bto\b|\band\b", "-", t)

    # Normalize common typed numbers/phrases to digits before numeric parsing
    word_num = {
        "one": "1",
        "two": "2",
        "three": "3",
        "four": "4",
        "five": "5",
        "six": "6",
        "seven": "7",
        "eight": "8",
        "nine": "9",
        "ten": "10",
        "eleven": "11",
        "twelve": "12",
    }
    for w, d in word_num.items():
        t = re.sub(rf"\b{w}\b", d, t)

    t = re.sub(r"\ba dozen\b|\bdozen\b", "12", t)
    t = re.sub(r"\ba hundred\b|\bhundred\b", "100", t)
    t = re.sub(r"\ba thousand\b|\bthousand\b", "1000", t)
    t = re.sub(r"\btens\b", "10", t)

    t = re.sub(r"\s+", " ", t).strip(" ,;:")

    # Numeric range patterns
    m_range = re.search(r"(?<!\d)(\d+)\s*-\s*(\d+)(?!\d)", t)
    if m_range:
        a, b = int(m_range.group(1)), int(m_range.group(2))
        lo, hi = sorted((a, b))
        return f"{lo}-{hi}"

    # Single number with relational symbols
    m_rel = re.search(r"(<=|>=|<|>|~)\s*(\d+)", t)
    if m_rel:
        op, n = m_rel.group(1), int(m_rel.group(2))
        return f"{op}{n}"

    # Bare single number
    m_num = re.fullmatch(r"(\d+)", t)
    if m_num:
        return m_num.group(1)

    # Coarse size words
    if re.search(r"\bdozens\b", t):
        return "dozens"
    if re.search(r"\bhundreds\b|\bfew hundred\b|\bseveral hundred\b", t):
        return "hundreds"
    if re.search(r"\bthousands\b|\bfew thousand\b|\bseveral thousand\b", t):
        return "thousands"

    # Fallback: remove punctuation noise and keep normalized text
    t = re.sub(r"[^a-z0-9<>~=+\- ]", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


def recode_numeric(canon: str):
    # User-requested explicit overrides
    if canon == "no report":
        return "unreported", "no report/unreported/unknown -> unreported"

    # A-B -> average of A and B
    m_range = re.fullmatch(r"(\d+)-(\d+)", canon)
    if m_range:
        a, b = int(m_range.group(1)), int(m_range.group(2))
        return round((a + b) / 2, 2), "A-B -> (A+B)/2"

    # up to A -> A
    m_upto = re.fullmatch(r"<=\s*(\d+)", canon)
    if m_upto:
        a = int(m_upto.group(1))
        return float(a), "up to A -> A"

    # at least A -> A
    m_atleast = re.fullmatch(r">=\s*(\d+)", canon)
    if m_atleast:
        a = int(m_atleast.group(1))
        return float(a), "at least A -> A"

    # more than/over A -> A*1.1
    m_more = re.fullmatch(r">\s*(\d+)", canon)
    if m_more:
        a = int(m_more.group(1))
        return round(a * 1.1, 2), "more than/over A -> A*1.1"

    # ~A -> A
    m_approx = re.fullmatch(r"~\s*(\d+)", canon)
    if m_approx:
        a = int(m_approx.group(1))
        return float(a), "~A -> A"

    # <A -> A
    m_less = re.fullmatch(r"<\s*(\d+)", canon)
    if m_less:
        a = int(m_less.group(1))
        return float(a), "<A -> A"

    # Bare A -> A
    m_single = re.fullmatch(r"(\d+)", canon)
    if m_single:
        a = int(m_single.group(1))
        return float(a), "A -> A"

    # coarse words recodes
    if canon == "dozens":
        return 50.0, "dozens -> 50"
    if canon == "hundreds":
        return 500.0, "hundreds -> 500"
    if canon == "thousands":
        return 5000.0, "thousands -> 5000"

    return None, "unmapped"


# Group every raw label by canonical label
groups = defaultdict(list)
for raw_label, count in raw_label_counter.items():
    canon = canonicalize_label(raw_label)
    groups[canon].append((raw_label, count))

# Sort all groups by total counts (descending)
all_groups = []
for canon, variants in groups.items():
    total = sum(c for _, c in variants)
    variants_sorted = sorted(variants, key=lambda x: x[1], reverse=True)
    all_groups.append((canon, total, variants_sorted))
all_groups.sort(key=lambda x: x[1], reverse=True)

# Flatten all groups to a CSV-friendly table, including recodes
rows_out = []
unmapped_canonical = []
for canon, total, variants in all_groups:
    recode_value, recode_rule = recode_numeric(canon)
    if recode_value is None:
        unmapped_canonical.append((canon, total, len(variants)))

    for raw_label, count in variants:
        rows_out.append({
            "canonical_label": canon,
            "group_total_count": total,
            "raw_label": raw_label,
            "raw_label_count": count,
            "num_variants_in_group": len(variants),
            "recode_value": recode_value,
            "recode_rule": recode_rule,
        })

# Resolve the project Data directory robustly from current kernel working directory.
cwd = Path.cwd()
if (cwd / "Data").exists():
    data_dir = cwd / "Data"
elif (cwd.parent / "Data").exists():
    data_dir = cwd.parent / "Data"
else:
    data_dir = Path("../Data")
out_path = data_dir / "identical_crowd_size_label_sets.csv"

with out_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "canonical_label",
            "group_total_count",
            "raw_label",
            "raw_label_count",
            "num_variants_in_group",
            "recode_value",
            "recode_rule",
        ],
    )
    writer.writeheader()
    writer.writerows(rows_out)

mapped_rows = sum(1 for r in rows_out if r["recode_value"] is not None)

print("=== Full Label Recoding Export ===")
print(f"Canonical groups exported: {len(all_groups)}")
print(f"Raw label rows exported: {len(rows_out)}")
print(f"Rows with recode assigned: {mapped_rows}")
print(f"Saved full summary to: {out_path}")

print("\nUnmapped canonical labels:")
if unmapped_canonical:
    for canon, total, nvar in sorted(unmapped_canonical, key=lambda x: x[1], reverse=True):
        print(f"- {canon} | total={total} | variants={nvar}")
else:
    print("- None")

print("\nPreview of top groups:")
for canon, total, variants in all_groups[:15]:
    recode_value, recode_rule = recode_numeric(canon)
    print(
        f"\n[{canon}] total={total} | variants={len(variants)} | recode={recode_value} ({recode_rule})"
    )
    for raw_label, count in variants[:8]:
        print(f"- {raw_label}: {count}")
    if len(variants) > 8:
        print(f"- ... ({len(variants)-8} more variants)")

=== Full Label Recoding Export ===
Canonical groups exported: 300
Raw label rows exported: 478
Rows with recode assigned: 440
Saved full summary to: /Users/charleszhu/Desktop/ACLED/Data/identical_crowd_size_label_sets.csv

Unmapped canonical labels:
- a few 100 | total=57 | variants=1
- a group | total=40 | variants=1
- many | total=24 | variants=1
- fewer than 100 | total=20 | variants=1
- several | total=17 | variants=1
- some | total=16 | variants=1
- a few | total=12 | variants=1
- a large number | total=10 | variants=1
- several 100 | total=10 | variants=1
- several 12 | total=7 | variants=1
- a few 1000 | total=5 | variants=1
- >= a few 100 | total=5 | variants=1
- a lot of | total=5 | variants=1
- a large group | total=4 | variants=1
- large group | total=4 | variants=1
- large | total=4 | variants=1
- several 1000 | total=3 | variants=1
- a big group | total=2 | variants=1
- ~- 100 | total=2 | variants=1
- a number | total=2 | variants=1
- ~- 20 | total=2 | variants=1
- a small

In [46]:
# Final pass: resolve remaining unmapped labels in the identical-label CSV
import csv
import re
from collections import defaultdict
from pathlib import Path

# Resolve the project Data directory robustly from current kernel working directory.
cwd = Path.cwd()
if (cwd / "Data").exists():
    data_dir = cwd / "Data"
elif (cwd.parent / "Data").exists():
    data_dir = cwd.parent / "Data"
else:
    data_dir = Path("../Data")

csv_path = data_dir / "identical_crowd_size_label_sets.csv"

with csv_path.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)
    fieldnames = reader.fieldnames or []

if not fieldnames:
    raise ValueError("CSV has no header. Run Cell 9 first.")


def normalize_unmapped_label(canon: str) -> str:
    t = (canon or "").strip().lower()
    t = re.sub(r"\s+", " ", t)

    # Fix common typos and malformed operator expressions.
    t = t.replace("las many as", "as many as")
    t = re.sub(r"^up to\s*(\d+)$", r"<=\1", t)
    t = re.sub(r"^as many as\s*(\d+)$", r"<=\1", t)
    t = re.sub(r"^(\d+)\s*>$", r">\1", t)
    t = re.sub(r"^~\s*-\s*(\d+)$", r"~\1", t)
    t = re.sub(r"^~\s*a\s*(\d+)$", r"~\1", t)
    t = re.sub(r"^<=\s*a\s*(\d+)$", r"<=\1", t)
    t = re.sub(r"^>=\s*a\s*(\d+)$", r">=\1", t)
    return t


def recode_extra(canon: str):
    t = normalize_unmapped_label(canon)

    # Rule 1: qualitative small labels -> 10
    to_10 = {
        "a few", "some", "many", "a group", "several", "few", "numerous",
        "a number", "a big group", "a small group", "large group", "a large group",
        ">= a few", ">= several"
    }
    if t in to_10:
        return 10.0, "qualitative_small -> 10"

    # Rule 2: qualitative large labels -> 100
    to_100 = {"a lot of", "a large number", "large number", "large"}
    if t in to_100:
        return 100.0, "qualitative_large -> 100"

    # Rule 3: "a few + XXX" -> XXX * 3
    m_few = re.fullmatch(r"(?:>=\s*|<=\s*|~\s*)?a few\s+(\d+)", t)
    if m_few:
        n = int(m_few.group(1))
        return float(n * 3), "a few + XXX -> 3*XXX"

    # Extend same pattern for "several + XXX" -> XXX * 3
    m_several = re.fullmatch(r"(?:>=\s*|<=\s*|~\s*)?several\s+(\d+)", t)
    if m_several:
        n = int(m_several.group(1))
        return float(n * 3), "several + XXX -> 3*XXX"

    # Normalize remaining numeric/operator expressions.
    m_upto = re.fullmatch(r"<=\s*(\d+)", t)
    if m_upto:
        return float(int(m_upto.group(1))), "up to A -> A"

    m_atleast = re.fullmatch(r">=\s*(\d+)", t)
    if m_atleast:
        return float(int(m_atleast.group(1))), "at least A -> A"

    m_more = re.fullmatch(r">\s*(\d+)", t)
    if m_more:
        a = int(m_more.group(1))
        return round(a * 1.1, 2), "more than/over A -> A*1.1"

    m_less = re.fullmatch(r"<\s*(\d+)", t)
    if m_less:
        return float(int(m_less.group(1))), "<A -> A"

    m_approx = re.fullmatch(r"~\s*(\d+)", t)
    if m_approx:
        return float(int(m_approx.group(1))), "~A -> A"

    # Dataset-specific text noise with no crowd number.
    if t == "jilin chemical fibre group":
        return "unreported", "non-crowd text -> unreported"

    if t == "fewer than 100":
        return 100.0, "fewer than 100 -> 100"

    return None, "unmapped"


before_unmapped = 0
updated_rows = 0
for row in rows:
    rule = (row.get("recode_rule") or "").strip().lower()
    val = (row.get("recode_value") or "").strip()

    if rule == "unmapped" or val == "":
        before_unmapped += 1
        new_value, new_rule = recode_extra(row.get("canonical_label", ""))
        if new_value is not None:
            row["recode_value"] = str(new_value)
            row["recode_rule"] = new_rule
            updated_rows += 1

with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

remaining = defaultdict(int)
for row in rows:
    rule = (row.get("recode_rule") or "").strip().lower()
    if rule == "unmapped":
        canon = (row.get("canonical_label") or "").strip()
        remaining[canon] += 1

print("=== Final Unmapped Cleanup Applied ===")
print(f"CSV updated: {csv_path}")
print(f"Rows that were unmapped before this pass: {before_unmapped}")
print(f"Rows recoded in this pass: {updated_rows}")
print(f"Rows still unmapped: {sum(remaining.values())}")
print(f"Canonical labels still unmapped: {len(remaining)}")

print("\nRemaining unmapped labels:")
if remaining:
    for canon, n in sorted(remaining.items(), key=lambda x: (-x[1], x[0])):
        print(f"- {canon}: {n}")
else:
    print("- None")

=== Final Unmapped Cleanup Applied ===
CSV updated: /Users/charleszhu/Desktop/ACLED/Data/identical_crowd_size_label_sets.csv
Rows that were unmapped before this pass: 38
Rows recoded in this pass: 38
Rows still unmapped: 0
Canonical labels still unmapped: 0

Remaining unmapped labels:
- None


In [47]:
# Add a 'Crowd Size' column to cleaned_data.csv using the finalized recoding rules
import csv
import re
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from pathlib import Path


# Reuse the same normalization logic so labels from Tags match the recoding table.
def canonicalize_label(label: str) -> str:
    t = label.lower().strip()
    t = t.replace("–", "-").replace("—", "-")
    t = re.sub(r"\s+", " ", t)

    if t in {
        "no report",
        "unreported",
        "not reported",
        "none reported",
        "unknown",
        "no reports",
        "no reported",
        "-no report",
        "unstated",
    }:
        return "no report"

    t = re.sub(r"\bbetween\b", "", t)
    t = re.sub(r"\bfrom\b", "", t)
    t = re.sub(r"\babout\b|\baround\b|\bapproximately\b|\bapprox\.?\b|\balmost\b|\bnearly\b", "~", t)
    t = re.sub(r"\bclose\s*-?\s*", "~", t)
    t = re.sub(r"\bat least\b", ">=", t)
    t = re.sub(r"\bmore than\b|\bover\b|\bor more\b", ">", t)
    t = re.sub(r"\bless than\b", "<", t)
    t = re.sub(r"\bup to\b", "<=", t)
    t = re.sub(r"\bas many as\b", "<=", t)
    t = re.sub(r"\bto\b|\band\b", "-", t)

    word_num = {
        "one": "1",
        "two": "2",
        "three": "3",
        "four": "4",
        "five": "5",
        "six": "6",
        "seven": "7",
        "eight": "8",
        "nine": "9",
        "ten": "10",
        "eleven": "11",
        "twelve": "12",
    }
    for w, d in word_num.items():
        t = re.sub(rf"\b{w}\b", d, t)

    t = re.sub(r"\ba dozen\b|\bdozen\b", "12", t)
    t = re.sub(r"\ba hundred\b|\bhundred\b", "100", t)
    t = re.sub(r"\ba thousand\b|\bthousand\b", "1000", t)
    t = re.sub(r"\btens\b", "10", t)
    t = re.sub(r"\s+", " ", t).strip(" ,;:")

    m_range = re.search(r"(?<!\d)(\d+)\s*-\s*(\d+)(?!\d)", t)
    if m_range:
        a, b = int(m_range.group(1)), int(m_range.group(2))
        lo, hi = sorted((a, b))
        return f"{lo}-{hi}"

    m_rel = re.search(r"(<=|>=|<|>|~)\s*(\d+)", t)
    if m_rel:
        op, n = m_rel.group(1), int(m_rel.group(2))
        return f"{op}{n}"

    m_num = re.fullmatch(r"(\d+)", t)
    if m_num:
        return m_num.group(1)

    if re.search(r"\bdozens\b", t):
        return "dozens"
    if re.search(r"\bhundreds\b|\bfew hundred\b|\bseveral hundred\b", t):
        return "hundreds"
    if re.search(r"\bthousands\b|\bfew thousand\b|\bseveral thousand\b", t):
        return "thousands"

    t = re.sub(r"[^a-z0-9<>~=+\- ]", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


def round_if_number(value: str) -> str:
    try:
        rounded = Decimal(value).quantize(Decimal("1"), rounding=ROUND_HALF_UP)
        return str(int(rounded))
    except (InvalidOperation, ValueError):
        return value


cwd = Path.cwd()
if (cwd / "Data").exists():
    data_dir = cwd / "Data"
elif (cwd.parent / "Data").exists():
    data_dir = cwd.parent / "Data"
else:
    data_dir = Path("../Data")

cleaned_path = data_dir / "cleaned_data.csv"
map_path = data_dir / "identical_crowd_size_label_sets.csv"
temp_path = cleaned_path.with_suffix(".tmp.csv")

if not cleaned_path.exists():
    raise FileNotFoundError(f"Missing file: {cleaned_path}")
if not map_path.exists():
    raise FileNotFoundError(f"Missing file: {map_path}")

# Load canonical -> recoded value map from the finalized recoding table.
canon_to_value = {}
with map_path.open(newline="", encoding="utf-8") as f:
    r = csv.DictReader(f)
    for row in r:
        canon = (row.get("canonical_label") or "").strip()
        val = (row.get("recode_value") or "").strip()
        if canon and val:
            canon_to_value[canon] = val

with cleaned_path.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)
    fields = reader.fieldnames or []

if not fields:
    raise ValueError("cleaned_data.csv has no header.")

field_map = {f.lower(): f for f in fields}
tags_col = field_map.get("tags")
if not tags_col:
    raise ValueError("No 'tags' column found in cleaned_data.csv")

out_fields = list(fields)
if "Crowd Size" not in out_fields:
    out_fields.append("Crowd Size")

crowd_pattern = re.compile(r"crowd\s*size\s*=\s*([^;|,]+)", re.IGNORECASE)

assigned = 0
blank = 0
not_found = 0

for row in rows:
    tag_text = (row.get(tags_col) or "").strip()
    matches = crowd_pattern.findall(tag_text)

    if not matches:
        row["Crowd Size"] = ""
        blank += 1
        continue

    # Use the first crowd-size mention in Tags.
    raw_label = " ".join(matches[0].split()).lower()
    canon = canonicalize_label(raw_label)
    val = canon_to_value.get(canon)

    if val is None:
        row["Crowd Size"] = ""
        not_found += 1
    else:
        row["Crowd Size"] = round_if_number(val)
        assigned += 1

with temp_path.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=out_fields, extrasaction="ignore")
    w.writeheader()
    w.writerows(rows)

temp_path.replace(cleaned_path)

print("=== Crowd Size Column Added ===")
print(f"Updated file: {cleaned_path}")
print(f"Rows total: {len(rows)}")
print(f"Rows assigned Crowd Size: {assigned}")
print(f"Rows without crowd-size mention in Tags: {blank}")
print(f"Rows with crowd-size mention but no map match: {not_found}")

=== Crowd Size Column Added ===
Updated file: /Users/charleszhu/Desktop/ACLED/Data/cleaned_data.csv
Rows total: 8937
Rows assigned Crowd Size: 8937
Rows without crowd-size mention in Tags: 0
Rows with crowd-size mention but no map match: 0


## 4. Text Pre-Processing
This section pre-process the text data in the 'Notes' column to facilitate automated text analysis. The procedures include tokenization, lemmatization, and stopwords removal.

In [48]:
from pathlib import Path

import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Use widely adopted NLTK resources for tokenization, lemmatization, and stopword lists.
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

# Resolve the cleaned_data.csv path first so we can load/save consistently.
cwd = Path.cwd()
if "cleaned_path" in globals():
    csv_path = Path(cleaned_path)
elif (cwd / "Data").exists():
    csv_path = cwd / "Data" / "cleaned_data.csv"
elif (cwd.parent / "Data").exists():
    csv_path = cwd.parent / "Data" / "cleaned_data.csv"
else:
    csv_path = Path("../Data/cleaned_data.csv")

# Fallback: load cleaned_data if it is not already available in memory.
if "cleaned_data" not in globals():
    cleaned_data = pd.read_csv(csv_path)

# Resolve Notes column name robustly (case/whitespace tolerant).
column_lookup = {str(c).strip().lower(): c for c in cleaned_data.columns}
if "notes" not in column_lookup:
    raise KeyError(f"Could not find a 'Notes' column. Available columns: {list(cleaned_data.columns)}")
notes_col = column_lookup["notes"]

base_stop_words = set(stopwords.words("english"))

# China's provinces and autonomous regions (pinyin romanization)
# Source: Official Chinese administrative divisions
china_provinces = {
    # Direct-controlled municipalities (直辖市)
    "beijing", "shanghai", "tianjin", "chongqing",
    # Provincial-level regions
    "hebei", "shanxi", "liaoning", "jilin", "heilongjiang",
    "jiangsu", "zhejiang", "anhui", "fujian", "jiangxi",
    "shandong", "henan", "hubei", "hunan", "guangdong", "guangxi",
    "hainan", "sichuan", "guizhou", "yunnan",
    "xizang",  # Tibet
    "shaanxi", "gansu", "qinghai", "ningxia", "xinjiang",
    "neimenggu",  # Inner Mongolia (also written as "nmg" or "inner_mongolia")
}

# China's prefecture-level cities (地级市) and administrative divisions
# Source: Official China administrative division codes (State Council data)
# Includes all prefecture-level cities from 31 provinces/regions/autonomous regions
china_prefecture_cities = {
    # Direct-controlled municipalities (直辖市)
    "beijing", "shanghai", "tianjin", "chongqing",
    # Hebei (河北) - 11 cities
    "shijiazhuang", "tangshan", "qinhuangdao", "handan", "xingtai", "baoding",
    "zhangjiakou", "chengde", "cangzhou", "langfang", "hengshui",
    # Shanxi (山西) - 11 cities
    "taiyuan", "datong", "yangquan", "changzhi", "jincheng", "shuozhou",
    # Liaoning (辽宁) - 14 cities
    "shenyang", "dalian", "anshan", "fushun", "benxi", "dandong", "liaoyang", "panjin",
    "tieling", "chaoyang",
    # Jilin (吉林) - 8 cities
    "changchun", "jilin", "siping", "liaoyang", "tonghua", "baishan", "songyuan", "baicheng",
    # Heilongjiang (黑龙江) - 12 cities
    "harbin", "qiqihar", "heihe", "suihua", "daqing", "yichun", "hegang", "mudanjiang",
    "jiamusi", "shuangyashan",
    # Jiangsu (江苏) - 13 cities
    "nanjing", "xuzhou", "changzhou", "suzhou", "nantong", "lianyungang", "yangzhou",
    "zhenjiang", "taizhou", "wuxi", "suqian", "yancheng",
    # Zhejiang (浙江) - 11 cities
    "hangzhou", "ningbo", "wenzhou", "jiaxing", "huzhou", "shaoxing", "jinhua", "quzhou",
    "zhoushan", "lishui",
    # Anhui (安徽) - 16 cities
    "hefei", "wuhu", "bengbu", "huainan", "maanshan", "huaibei", "tongling", "anqing",
    "chuzhou", "fuyang", "liuan", "chaohu", "chizhou", "xuancheng", "bozhou",
    # Fujian (福建) - 9 cities
    "fuzhou", "xiamen", "putian", "sanming", "quanzhou", "zhangzhou", "nanping", "longyan",
    # Jiangxi (江西) - 11 cities
    "nanchang", "jingdezhen", "pingxiang", "jiujiang", "xinyu", "ganzhou", "jian",
    "yingtan", "shangrao", "fuzhou",
    # Shandong (山东) - 17 cities
    "jinan", "qingdao", "zibo", "zaozhuang", "dongying", "yantai", "weifang", "jining",
    "taishan", "weihai", "rizhao", "laiwu", "linyi", "dezhou", "liaocheng", "binzhou", "heze",
    # Henan (河南) - 18 cities
    "zhengzhou", "kaifeng", "luoyang", "pingdingshan", "anyang", "hebi", "xinxiang",
    "jiaozuo", "puyang", "xuchang", "luohe", "sanmenxia", "nanyang", "shangqiu", "xinyang", "zhoukou", "zhumadian",
    # Hubei (湖北) - 13 cities
    "wuhan", "huangshi", "shiyan", "yichang", "xiangyang", "ezhou", "xiaogan", "jingzhou",
    "huanggang", "xianning", "enshi",
    # Hunan (湖南) - 14 cities
    "changsha", "zhuzhou", "xiangtan", "hengyang", "shaoyang", "yueyang", "changde",
    "yiyang", "zhangjiajie", "loudi",
    # Guangdong (广东) - 21 cities
    "guangzhou", "shenzhen", "zhuhai", "shantou", "foshan", "jiangmen", "zhanjiang",
    "maoming", "zhaoqing", "huizhou", "meizhou", "shanwei", "heyuan", "yangjiang",
    "qingyuan", "dongguan", "zhongshan", "chaozhou", "jieyang",
    # Guangxi (广西) - 14 cities
    "nanning", "liuzhou", "guilin", "wuzhou", "beihai", "fangchengang", "qinzhou",
    "guigang", "yulin", "baise", "hezhou", "hechi", "laibin", "chongzuo",
    # Hainan (海南) - 4 cities
    "haikou", "sanya", "sansha", "danzhou",
    # Sichuan (四川) - 21 cities
    "chengdu", "zigong", "luzhou", "deyang", "mianyang", "guangyuan", "suining",
    "neijiang", "leshan", "nanchong", "meishan", "yibin", "guangan", "dazhou", "bazhong", "yaan",
    # Guizhou (贵州) - 9 cities
    "guiyang", "liupanshui", "zunyi", "anshun", "bijie", "tongren",
    # Yunnan (云南) - 16 cities
    "kunming", "qujing", "yuxi", "baoshan", "zhaotong", "lijiang", "pu", "lincang",
    "wenshan", "honghe", "dali", "chuxiong",
    # Tibet (西藏) - 6 cities
    "lhasa", "rikaze", "linzhi", "shannan", "chamdo", "nagqu",
    # Shaanxi (陕西) - 10 cities
    "xian", "tongchuan", "baoji", "xiangyang", "weinan", "yanan", "hanzhong", "yulin", "shangluo",
    # Gansu (甘肃) - 14 cities
    "lanzhou", "jiayuguan", "jinchang", "baiyin", "tianshui", "wuwei", "zhangye",
    "pingliang", "jiuquan", "qingyang", "longnan", "linxia", "gannan",
    # Qinghai (青海) - 8 cities
    "xining", "haidong", "haibei", "huangnan", "yushu", "guoluo", "haixi",
    # Ningxia (宁夏) - 5 cities
    "yinchuan", "shizuishan", "wuzhong", "guyuan", "zhongwei",
    # Xinjiang (新疆) - 15 cities
    "urumqi", "karamay", "turpan", "hami", "changji", "bortala", "bayingolin",
    "aksu", "kashgar", "hotan", "tacheng", "altay",
    # Inner Mongolia (内蒙古) - 12 prefecture-level divisions
    "huhhot", "baotou", "wuhai", "chifeng", "tongliao", "ordos", "hulunbeier",
    "bayan", "ulanchabu", "xingan", "xilin", "alxa",
}

custom_stop_words = {
    "january", "february", "march", "april", "may", "june",
    "july", "august", "september", "october", "november", "december",
    "around", "several", "dozen", "dozens", "thousand", "thousands", "hundred", "hundreds",
    "reported", "demand", "protest", "protests", "protested", "demonstrate", "staged", "gather",
    "outside", "many", "coded", "held", "district", "county", "protester", "demonstarted", "go", "hold",
    "report", "cod", "hold", "stag", "province", "prefecture", "cpunty", "city", "town", "village", 
    "area", "region", "group", "groups",
}

# Union all stopwords: NLTK base + provinces + prefecture-level cities + custom terms
stop_words = base_stop_words.union(custom_stop_words).union(china_provinces).union(china_prefecture_cities)
lemmatizer = WordNetLemmatizer()


def preprocess_text(text: object) -> str:
    if pd.isna(text):
        return ""

    tokens = word_tokenize(str(text).lower())
    cleaned_tokens = [tok for tok in tokens if tok.isalnum() and not any(ch.isdigit() for ch in tok)]

    # Lemmatize first, then apply stopword removal on normalized lemmas.
    lemmatized_tokens = [lemmatizer.lemmatize(lemmatizer.lemmatize(tok, pos="v"), pos="n") for tok in cleaned_tokens]
    filtered_tokens = [tok for tok in lemmatized_tokens if tok not in stop_words]
    return " ".join(filtered_tokens)


processed_series = cleaned_data[notes_col].apply(preprocess_text)

# Ensure Processed_text is placed directly to the right of Notes.
if "Processed_text" in cleaned_data.columns:
    cleaned_data = cleaned_data.drop(columns=["Processed_text"])

notes_idx = cleaned_data.columns.get_loc(notes_col)
cleaned_data.insert(notes_idx + 1, "Processed_text", processed_series)

# Persist changes to cleaned_data.csv.
cleaned_data.to_csv(csv_path, index=False, encoding="utf-8")

print("Re-processed Notes using NLTK tokenization + lemmatization + stopwords.")
print("Included all China provinces and prefecture-level cities as stopwords.")
print("Overwrote 'Processed_text' and saved to cleaned_data.csv.")
print(f"Saved file: {csv_path}")
print(cleaned_data[[notes_col, "Processed_text"]].head())

Re-processed Notes using NLTK tokenization + lemmatization + stopwords.
Included all China provinces and prefecture-level cities as stopwords.
Overwrote 'Processed_text' and saved to cleaned_data.csv.
Saved file: /Users/charleszhu/Desktop/ACLED/Data/cleaned_data.csv
                                               notes  \
0  Around 1 January 2018 (as reported), between 1...   
1  Around 1 January 2018 (as reported), between 1...   
2  Around 1 January 2018 (as reported), between 1...   
3  Around 1 January 2018 (as reported), between 1...   
4  On 2 January 2018, more than 500 public primar...   

                                      Processed_text  
0       worker wage arrears owe coal company beipiao  
1  worker block entrance restaurant truck wage ar...  
2  worker wage arrears owe department store yongzhou  
3          worker wage arrears real estate developer  
4  public primary secondary school teacher main s...  
